In [103]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.utils import shuffle

In [105]:
data = pd.read_csv("/content/emails.csv")

texts = data["text"].astype(str).values
labels = data["spam"].values

In [106]:
def preprocess(text):

    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()

    return tokens


processed_texts = [preprocess(t) for t in texts]

In [107]:
def build_vocab(texts, min_freq=2):

    counter = Counter()

    for text in texts:
        counter.update(text)

    vocab = {}
    idx = 0

    for word, count in counter.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1

    return vocab


vocab = build_vocab(processed_texts)

print("Vocabulary size:", len(vocab))

Vocabulary size: 20351


In [108]:
def compute_tf(text, vocab):

    vec = np.zeros(len(vocab))

    for word in text:
        if word in vocab:
            vec[vocab[word]] += 1

    if len(text) > 0:
        vec = vec / len(text)

    return vec

In [109]:
def compute_idf(texts, vocab):

    N = len(texts)
    df = np.zeros(len(vocab))

    for text in texts:
        unique_words = set(text)

        for word in unique_words:
            if word in vocab:
                df[vocab[word]] += 1

    idf = np.log((N + 1) / (df + 1)) + 1

    return idf

idf = compute_idf(processed_texts, vocab)

In [110]:
X = np.array([compute_tf(text, vocab) for text in processed_texts])
X = X * idf

y = labels
X, y = shuffle(X, y, random_state=42)

In [111]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [113]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [114]:
class LogisticRegressionScratch:

    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):

        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        for epoch in range(self.epochs):

            linear = np.dot(X, self.weights) + self.bias
            preds = sigmoid(linear)

            dw = (1/n_samples) * np.dot(X.T, (preds - y))
            db = (1/n_samples) * np.sum(preds - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if epoch % 200 == 0:
                loss = -np.mean(
                    y*np.log(preds + 1e-9) +
                    (1-y)*np.log(1-preds + 1e-9)
                )
                print(f"Epoch {epoch} Loss: {loss}")

    def predict(self, X):

        linear = np.dot(X, self.weights) + self.bias
        probs = sigmoid(linear)

        return (probs >= 0.5).astype(int)

In [115]:
model = LogisticRegressionScratch(lr=0.5, epochs=8000)

model.fit(X_train, y_train)

Epoch 0 Loss: 0.6931471785599453
Epoch 200 Loss: 0.5250197902365804
Epoch 400 Loss: 0.49864573997458383
Epoch 600 Loss: 0.4748276189452205
Epoch 800 Loss: 0.453289467361569
Epoch 1000 Loss: 0.43376899857496815
Epoch 1200 Loss: 0.41602776281712606
Epoch 1400 Loss: 0.3998544388812461
Epoch 1600 Loss: 0.3850642236880912
Epoch 1800 Loss: 0.37149652112725895
Epoch 2000 Loss: 0.35901211310908476
Epoch 2200 Loss: 0.34749035012675955
Epoch 2400 Loss: 0.336826574837447
Epoch 2600 Loss: 0.32692984550725046
Epoch 2800 Loss: 0.31772096300991526
Epoch 3000 Loss: 0.3091307792839148
Epoch 3200 Loss: 0.3010987560230771
Epoch 3400 Loss: 0.2935717407458882
Epoch 3600 Loss: 0.28650292910332853
Epoch 3800 Loss: 0.2798509853785757
Epoch 4000 Loss: 0.2735792966421618
Epoch 4200 Loss: 0.26765533948074816
Epoch 4400 Loss: 0.26205014139575905
Epoch 4600 Loss: 0.25673782178228227
Epoch 4800 Loss: 0.2516951998321039
Epoch 5000 Loss: 0.24690145877663103
Epoch 5200 Loss: 0.24233785763138457
Epoch 5400 Loss: 0.2379

In [116]:
preds = model.predict(X_test)

accuracy = np.mean(preds == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.9380453752181501


In [117]:
def predict_email(email):

    tokens = preprocess(email)

    vec = compute_tf(tokens, vocab) * idf

    pred = model.predict(vec.reshape(1,-1))[0]

    if pred == 1:
        print("Spam Email")
    else:
        print("Not Spam")


predict_email("Your One Time Password (OTP) for login: 4155987.OTP is valid only for 05:00 mins. Do not share this OTP with anyone.")
predict_email("Find Your Gang on Travel Buddy! Connect with travelers from around the world! Chat, plan, and book your next adventure together. And here’s the exciting part… One Lucky Woman Travels FREE Every Month also cash prize!All you have to do is:1️⃣ Post more & more Find Buddy post to Find your travel tribe on Travel Buddy2️⃣ Book your trip through us3️⃣ You automatically enter the Monthly Free Trip Draws")

Spam Email
Spam Email


In [118]:
print(data['spam'].value_counts())

spam
0    4360
1    1368
Name: count, dtype: int64


In [119]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds)
print(cm)

[[891   1]
 [ 70 184]]


In [120]:
test_data = pd.read_csv("/content/mail_data.csv")

test_texts = test_data["text"].astype(str).values
test_labels = test_data["spam"].values

In [121]:
processed_test_texts = [preprocess(t) for t in test_texts]

In [122]:
X_test_new = np.array([compute_tf(text, vocab) for text in processed_test_texts])
X_test_new = X_test_new * idf

In [123]:
preds_new = model.predict(X_test_new)

In [124]:
accuracy = np.mean(preds_new == test_labels)
print("New Dataset Accuracy:", accuracy)

New Dataset Accuracy: 0.7727925340990668


In [125]:
print(confusion_matrix(test_labels, preds_new))

[[3924  901]
 [ 365  382]]
